In [22]:
from pathlib import Path
import re
import torch
import pandas as pd

outputs_dir = Path("outputs")


def epoch_from_name(p: Path):
    # gérer les deux formats : epoch_XX et checkpoint_epoch_XX
    m = re.search(r"(?:checkpoint_)?epoch_(\d+)", p.stem)
    return int(m.group(1)) if m else None


def scalar(x):
    if isinstance(x, torch.Tensor):
        return float(x.item()) if x.numel() == 1 else None
    if isinstance(x, (int, float)):
        return float(x)
    return None


def extract_metrics(organ):
    ckpt_dir = outputs_dir / f"checkpoints_{organ}_attention"
    
    files = sorted(
        ckpt_dir.glob("*epoch_*.pth"),
        key=lambda p: epoch_from_name(p) if epoch_from_name(p) is not None else -1
    )

    rows = []
    for f in files:
        c = torch.load(f, map_location="cpu")

        epoch = int(c.get("epoch_1_based", epoch_from_name(f)))
        dice = scalar(c.get("avg_val_dice", c.get("best_val_dice")))
        loss = scalar(c.get("avg_train_loss"))

        rows.append({
            "organ": organ,
            "epoch": epoch,
            "dice": dice,
            "loss": loss
        })

    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame

    return frame.sort_values("epoch").reset_index(drop=True)


df_liver = extract_metrics("liver")
df_tumor = extract_metrics("tumor")

print("NaN liver:", df_liver[["dice", "loss"]].isna().sum().to_dict() if not df_liver.empty else {"dice": 0, "loss": 0})
print("NaN tumor:", df_tumor[["dice", "loss"]].isna().sum().to_dict() if not df_tumor.empty else {"dice": 0, "loss": 0})

display(df_liver)
display(df_tumor)

C:\Users\neifa\AppData\Local\Temp\ipykernel_32968\1671814834.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  c = torch.load(f, map_location="cpu")
C:\Users\neifa\AppDat

NaN liver: {'dice': 0, 'loss': 8}
NaN tumor: {'dice': 0, 'loss': 0}


,organ,epoch,dice,loss
0,liver,0,0.754923,None
1,liver,1,0.835327,None
2,liver,2,0.852023,None
3,liver,3,0.852023,None
4,liver,4,0.886843,None
5,liver,5,0.886843,None
6,liver,6,0.899633,None
7,liver,7,0.899633,None


,organ,epoch,dice,loss
0,tumor,1,0.218554,0.121287
1,tumor,2,0.529987,0.038188
2,tumor,3,0.815569,0.027248
3,tumor,4,0.715736,0.022770
4,tumor,5,0.777757,0.020880
5,tumor,6,0.832640,0.017670
6,tumor,7,0.736651,0.015792
7,tumor,8,0.773493,0.013997
8,tumor,9,0.764672,0.012716
9,tumor,10,0.762375,0.012531
